# Training di value + policy head su Kaggle (imitazione dal dataset BoardSpace)

Gemello di `train_colab.ipynb` per Kaggle Notebooks, aggiornato al dataset
**policy** (`boardspace_policy_<anno>.jsonl`): ogni posizione porta anche le
mosse legali con move features e target pi (1 sulla mossa umana). Si allena
DA ZERO, con le due teste insieme: MSE del value + cross-entropy della
policy (`--policy-weight`). A fine training il value deve tornare ai livelli
della gen-0 (val MSE ~0,68, segno corretto ~72%): e' il controllo che il
multi-task non lo ha danneggiato.
Produce **solo il checkpoint dei pesi**: l'export TorchScript per il C++ va
fatto in locale (`torch_geometric==2.6.1`), vedi `docs/spiegazione_value_network.md` §6.

### Prima di eseguire

1. **Account Kaggle verificato** (numero di telefono): serve per GPU e Internet.
2. **Crea un Dataset Kaggle privato** (*Datasets → New Dataset*) con tre file:
   `hive_value_gnn.py`, `train_hive_value_gnn.py` (versioni correnti dal repo)
   e `boardspace_policy_jsonl.zip`.
   NB: Kaggle **scompatta da solo** gli zip caricati — nel Dataset compariranno
   direttamente i `boardspace_policy_<anno>.jsonl` (47 GB scompattati: il
   limite per Dataset e' 200 GB, ci stanno).
3. **Nuovo Notebook** (*Code → New Notebook*), poi importa questo file
   (*File → Import Notebook*) e nel pannello di destra:
   - *Input → Add Input*: collega il dataset creato al passo 2;
   - *Session options → Accelerator*: **GPU T4 x2** (o P100);
   - *Session options → Internet*: **ON** (serve per `pip install`).
4. Esegui le celle in ordine. Per un run non presidiato usa *Save Version →
   Save & Run All*: esegue tutto in background e conserva l'output anche se
   il browser si chiude (la sessione interattiva invece muore con lui).

Rispetto al primo run policy (29 luglio: --sample 0.5, val MSE 0.705,
policy CE 2.84) questo allena sul dataset INTERO con augmentation D6:
il trainer ora carica i tensori in float16 e impacchetta le partite in
Batch PyG (~13,5 GB in RAM sui ~29 disponibili, niente --sample), e con
`--augment` ogni posizione riceve a ogni epoca una delle 12 simmetrie
della board esagonale (rotazioni e riflessioni: informazione vera che la
rete non puo' dedurre da sola). La validation resta non aumentata, quindi
le metriche sono confrontabili col run precedente.
La quota gratuita e' 30 ore di GPU a settimana.

In [ ]:
# Trova script e dati nel Dataset collegato come Input, a qualunque
# profondita'. NB: Kaggle scompatta da solo gli zip caricati, quindi di
# norma i .jsonl sono gia' estratti (e lo zip non esiste piu').
import glob

def find_one(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None

TRAIN_PY = find_one('train_hive_value_gnn.py')
MODEL_PY = find_one('hive_value_gnn.py')
assert TRAIN_PY and MODEL_PY, 'script .py non trovati: collega il Dataset dal pannello Input'

INPUT_JSONL = sorted(glob.glob('/kaggle/input/**/boardspace_policy_*.jsonl', recursive=True))
ZIP = find_one('boardspace_policy_jsonl.zip')
assert INPUT_JSONL or ZIP, 'nessun boardspace_policy_*.jsonl e nessuno zip nel Dataset collegato'

CHECKPOINT = '/kaggle/working/checkpoint_gen0_policy_aug.pt'
print(f"{len(INPUT_JSONL)} jsonl gia' estratti" if INPUT_JSONL else f'zip da scompattare: {ZIP}')

In [ ]:
# Verifica GPU e installa PyTorch Geometric.
# NB: qui la versione di torch_geometric e' libera perche' si fa solo training;
# il pin ==2.6.1 riguarda solo l'export TorchScript, che si fa in locale.
import torch
print('GPU disponibile:', torch.cuda.is_available(),
      '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'attiva la GPU nelle Session options!')

%pip install -q torch_geometric

In [ ]:
# Copia gli script in /kaggle/working e prepara la lista dei file dati.
# Se i .jsonl sono gia' estratti nell'input si leggono da li' (read-only va
# bene); se invece c'e' ancora lo zip, si scompatta su disco effimero
# (/kaggle/tmp): NON in /kaggle/working, altrimenti i 47 GB di JSONL
# finirebbero salvati come output del notebook a ogni versione.
import subprocess

subprocess.run(['cp', TRAIN_PY, MODEL_PY, '/kaggle/working/'], check=True)

if INPUT_JSONL:
    DATA_FILES = INPUT_JSONL
else:
    !mkdir -p /kaggle/tmp/dataset && unzip -o -q "{ZIP}" -d /kaggle/tmp/dataset
    DATA_FILES = sorted(glob.glob('/kaggle/tmp/dataset/**/*.jsonl', recursive=True))

print(len(DATA_FILES), 'file jsonl:')
for f in DATA_FILES:
    print(' ', f)

In [ ]:
# Training value + policy DA ZERO sul dataset INTERO, con augmentation D6.
# Niente --sample: con float16 + partite impacchettate il dataset completo
# occupa ~13,5 GB sui ~29 disponibili. Se la RAM soffrisse comunque,
# aggiungere --sample 0.8.
# --max-hours 10.5: cintura di sicurezza sul limite di sessione Kaggle
# (12h, oltre le quali l'output della versione va PERSO) - il training si
# ferma prima di un'epoca che sforerebbe, con il best checkpoint salvo.
# 18 epoche: il run interrotto era al plateau gia' a meta' corsa, e il
# cosine comprime il decadimento sulle epoche disponibili (T_max=--epochs):
# stesso lr finale, meno ore di GPU. Col cosine il best tende ad arrivare
# nelle ULTIME epoche (lr basso che rifinisce): e' normale, non un segnale
# che servivano piu' epoche.
# --lr-schedule cosine: il run interrotto mostrava la validation che
# OSCILLAVA dopo l'epoca ~15 (lr costante troppo alto vicino al minimo);
# il decadimento coseno (da 1e-3 a 1e-4) stabilizza le epoche finali.
# Lettura dei log: riferimenti da battere - run 29 luglio (--sample 0.5):
# val MSE 0.705, CE 2.84; run interrotto (intero+augment, epoca 15):
# val MSE 0.685, CE 2.827. L'obiettivo e' scendere
# sotto entrambi; la CE parte da ~4,1 (uniforme su ~60 mosse legali).
files = ' '.join(DATA_FILES)

!cd /kaggle/working && python -u train_hive_value_gnn.py {files} \
    --output "{CHECKPOINT}" \
    --policy-weight 1.0 --augment --lr-schedule cosine \
    --max-hours 10.5 \
    --device cuda --batch-size 256 --epochs 18

### Dopo il training

Il checkpoint migliore è in `/kaggle/working/checkpoint_gen0_policy_aug.pt`: si
scarica dal file browser del pannello di destra (sessione interattiva) o
dalla scheda *Output* del notebook (dopo *Save & Run All*). Conviene anche
caricarlo nella cartella Drive `HiveGotThis_colab/`, dove stanno gli altri
checkpoint. In locale:

```bash
# esporta per il C++ (ambiente locale con torch_geometric==2.6.1)
python3 scripts/export_hive_value_gnn.py --weights checkpoint_gen0_policy_aug.pt --output hive_policy_gen0.pt

# il motore la usa (l'MCTS rileva la policy head e passa a PUCT da solo)
./build/HiveEngine hive_policy_gen0.pt
```

I passi successivi sono il benchmark del costo PUCT (la policy aggiunge una
chiamata alla rete per espansione) e il match dell'harness contro la gen-0
value-only: stesso match, stesse condizioni, un modello per lato
(`tools/uhp_match.py`, doc §8).